# LIME explanations - step 1: a few test images

**LIME** (Ribeiro et al., 2016) explains one prediction of a trained model: which parts of the image did the
model rely on? It splits the image into **superpixels** (SLIC), hides random subsets of them in 1000 copies of
the image, asks the model about every copy, and fits a weighted linear model from "which superpixels are
visible" to "probability of the class". The weight of a superpixel is its **importance**.
The LIME code is our own (`src/explain.py`); SLIC comes from scikit-image.

**Step 1 (this notebook):** the best model, ConvNeXt-Tiny, on 8 test images - 4 wrong predictions and, for each,
a correct one of the same material from another scene. For a wrong prediction both the predicted and the true
class are explained. Only figures and a small table; no measurements yet.

* The model is the analysed run of the training notebook (best validation accuracy), weights from `outputs/`
  or Google Drive.
* The images are the test images exactly as the model sees them (whole image, 512x512).
* The bounding boxes are not used.
* Outputs: `outputs/lime/<model>/`, saved to `results/lime/<model>/` on GitHub (and Google Drive).

**Run on a GPU** (university computer or Colab): about 2 seconds per explanation. On a CPU about 10 minutes.

## 1. Setup: code and data

In [1]:
# Works in Colab and on an own computer with an NVIDIA GPU (e.g. the university computer).
import os, sys
REPO_URL = "https://github.com/Ahmed-AlTahan/CNN-LIME.git"

if "google.colab" in sys.modules:          # Colab: get the code from GitHub
    REPO_DIR = "/content/thesis"
    if not os.path.isdir(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
else:                                      # own computer: this notebook is in <repository>/notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
!git -C "{REPO_DIR}" pull -q --ff-only
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import env
env.ensure_dataset()      # downloads the dataset into data/ the first time
env.print_info()

Dataset found: /content/thesis/data/15workpieces_same_position/all_images
Running on: colab | GPU: Tesla T4 | code: commit e49f109


## 2. Settings
The LIME settings come from `config.LIME`; change them here for an experiment.

In [2]:
import config, data, explain, env

MODEL = "convnext_tiny"          # the model to explain (another trained model: "resnet50", ...)
settings = config.LIME
# e.g.: settings = config.replace(config.LIME, n_segments=400, n_samples=2000)
out_dir = explain.output_dir(MODEL)
print(settings)

LimeSettings(n_segments=250, compactness=10.0, n_samples=1000, batch_size=50, kernel_width=0.25, ridge_alpha=1.0, top_k=5, seed=0)


## 3. Load the trained model
The run with the best validation accuracy, the same one analysed in the training notebook. If its weights are not on this computer, they are taken from Google Drive.

In [3]:
model, seed = explain.load_trained(MODEL)

convnext_tiny, seed 1: validation 96.53%, test 96.80% | weights: outputs/convnext_tiny/weights/convnext_tiny_seed1.pth


## 4. Choose the test images
From the test predictions of that run: 4 wrong predictions (metal confused with metal first, then black plastic), each paired with a correct image of the same material from another scene.

In [4]:
examples = explain.choose_examples(MODEL)

 # | kind    | scene | true           | predicted      | image
 1 | correct |    90 | aluminium      | aluminium      | 2024-04-25-123720.jpg
 2 | wrong   |    31 | aluminium      | steel          | 2024-04-18-104638.jpg
 3 | correct |   117 | steel          | steel          | 2024-04-29-102030.jpg
 4 | wrong   |   119 | steel          | aluminium      | 2024-04-29-102854.jpg
 5 | correct |    14 | steel          | steel          | 2024-04-15-095121.jpg
 6 | wrong   |    94 | steel          | black plastic  | 2024-04-25-130612.jpg
 7 | correct |    60 | black plastic  | black plastic  | 2024-04-22-103632.jpg
 8 | wrong   |   150 | black plastic  | red plastic    | 2024-05-03-112757.jpg


## 5. How LIME works, step by step (one image)
Before explaining all 8 images, the first steps of LIME on one of them. The image is example 4 (steel predicted
as aluminium); any example can be used by changing the number.

### Step 1: the image is split into superpixels
SLIC groups neighbouring pixels of similar colour into regions. These **superpixels are the features of LIME**:
LIME can only say which superpixels were important, never single pixels. Right: every superpixel in its mean
colour, which shows how coarse these features are.

In [5]:
demo = examples[3]                       # example 4 (list index starts at 0)
image = data.load_image(demo["img_path"])
segments = explain.show_superpixels(image, settings, out_dir)

AttributeError: module 'explain' has no attribute 'show_superpixels'

### Steps 2 and 3: random copies of the image, and the model's answer for each
LIME makes 1000 copies of the image. In every copy a random half of the superpixels is **hidden** (filled with
the mean colour of the whole image), and the model gives its probability for the class. Shown here: the original
and the first 4 of these copies, exactly as LIME uses them. When hiding a superpixel lowers the probability,
that superpixel was important - steps 4 and 5 (next section) turn the 1000 answers into one importance per
superpixel with a weighted linear model.

In [ ]:
target = demo["pred"]                    # explain the class the model predicted
probs = explain.show_variations(model, image, segments, target, settings, out_dir)

## 6. Explain them (steps 4 and 5)
One figure per image:
* **input**: the image as the model sees it, **superpixels**: the SLIC regions LIME switches on and off
* **top 5 superpixels**: the regions that support the explained class most (everything else is faded)
* **importance**: red = supports the class, blue = speaks against it
* **p**: the model's probability for that class; **fidelity R2**: how well LIME's linear model imitates the
  network around this image (close to 1 = the explanation is trustworthy)

For a wrong prediction the second row explains the **true** class: what would have spoken for the right answer.

In [ ]:
rows = explain.explain_examples(model, examples, settings, out_dir, MODEL)

## 7. Save the results
Figures and the table go to `results/lime/<model>/` on GitHub (and to `MyDrive/thesis_results/lime/<model>/` on Google Drive). A re-run replaces them.

In [ ]:
env.save_results(out_dir)